# Gemma2-9B-IT Local Feature Homeostasis (LFH) on BFCL Tool Calling

This notebook implements the BFCL Tool Calling LFH experiment using **Gemma2-9B-IT**, including the controlled source-block variation, query-wise LFH feature ranking, attention-head negative-write analysis, attention-head causal intervention, and MLP control.

The complete ordered list is rendered in one native user turn using the model-specific chat template.

- Model: `google/gemma-2-9b-it`
- Target layer: `21`
- SAE: `google/gemma-scope-9b-pt-res` / `layer_21/width_16k/average_l0_129/params.npz`
- Default block scoring: `[-1, -2]` relative to each semantic-block end

In [ ]:

import os
import re
import json
import math
import random
import warnings
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple, Sequence, Union

# Silence the harmless tqdm/Jupyter widget warning before importing datasets.
warnings.filterwarnings("ignore", message=r"IProgress not found.*")

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from scipy.stats import pearsonr


# ============================================================
# 0. Reproducibility and experiment config
# ============================================================

def set_seed(seed: int):
    print("seed:", seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(42)

# Model files for gated Gemma/Llama checkpoints are read from the existing
# local cache. Public Qwen checkpoints can fall back to an anonymous download.
# Public SAE files are downloaded anonymously when absent from the cache.
MODEL_LOCAL_FILES_ONLY = True
SAE_LOCAL_FILES_ONLY = False
MODEL_TRUST_REMOTE_CODE = False

# Model and SAE configuration.
REQUESTED_MODEL_NAME = 'Gemma2-9B-IT'
MODEL_NAME = 'google/gemma-2-9b-it'
MODEL_SHORT_NAME = 'Gemma-2-9B-IT'
TARGET_LAYER = 21
SAE_BACKEND = 'gemma_scope_npz'

SAE_REPO_ID = 'google/gemma-scope-9b-it-res'
SAE_TRAINER = None
SAE_BASE_DIR = None
SAE_WIDTH = 'width_16k'
SAE_AVERAGE_L0 = 'average_l0_47'
SAE_FILENAME = 'layer_20/width_16k/average_l0_47/params.npz'
SAE_HYPERPARAMS_FILENAME = None
SAE_LENS_RELEASE = None
SAE_LENS_SAE_ID = None
SAE_TOPK_K = None

EXPECTED_MODEL_NUM_LAYERS = 42
EXPECTED_D_MODEL = 3584
EXPECTED_D_SAE = 16384

# Task/list configuration.
TASK_TYPE = 'bfcl_tool'
ITEM_WORD = 'Tool'
ITEM_WORD_LOWER = 'tool'
ITEM_WORD_PLURAL = 'tools'
PROMPT_DATASET_NAME = 'BFCL Tool Calling'

# One native chat user turn contains the complete ordered list.
PROMPT_FORMAT = 'gemma_chat_single_user'
PROMPT_BOS_TOKEN = '<bos>'
TOKENIZE_ADD_SPECIAL_TOKENS = False
CHAT_TEMPLATE_KWARGS = {}
CHAT_TEMPLATE_USE_SYSTEM_ROLE = False

# High-norm-token filtering is used for the Qwen BatchTopK SAE suite only.
FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION = False
HIGH_NORM_FILTER_MULTIPLIER = 10.0
QWEN_HIGH_NORM_FILTER_MULTIPLIER = HIGH_NORM_FILTER_MULTIPLIER

DTYPE = torch.bfloat16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 30 fixed target blocks x 50 source-block variants by default.
NUM_TEST_QUERY = 30
NUM_PREVIOUS = 50
NUM_BLOCKS_TOTAL = 8
FREQ_THRESHOLD = 0.05

# LFH source/target block configuration.
LFH_TARGET_BLOCK_NUM = 8
LFH_SOURCE_BLOCK_NUM = 7
LFH_LOCALITY_SWEEP_SOURCE_BLOCK_NUMS = [7, 6, 5]


def block_num_to_index(block_num: int, total_blocks: int = NUM_BLOCKS_TOTAL) -> int:
    """Convert a 1-indexed semantic-block number to a Python list index."""
    block_num = int(block_num)
    if block_num < 1 or block_num > int(total_blocks):
        raise ValueError(f"block_num must be in [1, {total_blocks}], got {block_num}.")
    return block_num - 1


LFH_SOURCE_BLOCK_INDEX = block_num_to_index(LFH_SOURCE_BLOCK_NUM)
LFH_TARGET_BLOCK_INDEX = block_num_to_index(LFH_TARGET_BLOCK_NUM)

# For BFCL/NQ list prompts, block boundaries are recovered from Tool/Document
# headers rather than from the enclosing chat-turn terminator.
BLOCK_END_TOKEN = None
BLOCK_SCORE_MODE = "All" # "All" or "selected_offsets_from_end"
BLOCK_TOKEN_OFFSETS_FROM_END = [-1, -2]
# Alternative whole-block setting:
# BLOCK_SCORE_MODE = "All"
# BLOCK_TOKEN_OFFSETS_FROM_END = None
BLOCK_LAST_K_TOKENS = 1

QWEN_SYSTEM_PROMPT = "You are a careful assistant."
LLAMA_SYSTEM_PROMPT = "You are a careful assistant."
SYSTEM_PROMPT = (
    f"You are a careful assistant. You will read an ordered list of {ITEM_WORD_PLURAL}. "
    f"Track the sequence of {ITEM_WORD_PLURAL} carefully."
)

NOTE_FOR_THIS_VARIANT = 'LFH BFCL Tool Calling variant for Gemma2-9B-IT. Uses the layer-21 Gemma Scope 9B pretrained-model residual SAE (width 16k, average L0 129); all eight ordered tools are rendered in one native user turn.'
print(NOTE_FOR_THIS_VARIANT)


# ============================================================
# 1. Load model, tokenizer, and SAE
# ============================================================

def load_tokenizer_configured(model_name: str):
    common_kwargs = dict(
        trust_remote_code=MODEL_TRUST_REMOTE_CODE,
        use_fast=True,
        token=False,
    )
    if MODEL_LOCAL_FILES_ONLY:
        print("Loading tokenizer from local cache:", model_name)
        return AutoTokenizer.from_pretrained(
            model_name,
            local_files_only=True,
            **common_kwargs,
        )

    try:
        print("Loading tokenizer from local cache:", model_name)
        return AutoTokenizer.from_pretrained(
            model_name,
            local_files_only=True,
            **common_kwargs,
        )
    except Exception as local_error:
        print("Local tokenizer cache was incomplete; retrying an anonymous public download.")
        print("Local-cache error:", repr(local_error))
        return AutoTokenizer.from_pretrained(
            model_name,
            local_files_only=False,
            **common_kwargs,
        )


def load_model_configured(model_name: str):
    common_kwargs = dict(
        dtype=DTYPE,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=MODEL_TRUST_REMOTE_CODE,
        low_cpu_mem_usage=True,
        token=False,
    )
    if MODEL_LOCAL_FILES_ONLY:
        print("Loading model from local cache:", model_name)
        return AutoModelForCausalLM.from_pretrained(
            model_name,
            local_files_only=True,
            **common_kwargs,
        )

    try:
        print("Loading model from local cache:", model_name)
        return AutoModelForCausalLM.from_pretrained(
            model_name,
            local_files_only=True,
            **common_kwargs,
        )
    except Exception as local_error:
        print("Local model cache was incomplete; retrying an anonymous public download.")
        print("Local-cache error:", repr(local_error))
        return AutoModelForCausalLM.from_pretrained(
            model_name,
            local_files_only=False,
            **common_kwargs,
        )


print("Loading tokenizer:", MODEL_NAME)
tokenizer = load_tokenizer_configured(MODEL_NAME)
if not getattr(tokenizer, "is_fast", False):
    raise ValueError("A fast tokenizer is required because this notebook uses return_offsets_mapping=True.")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading model:", MODEL_NAME)
model = load_model_configured(MODEL_NAME)
model.eval()


def get_model_input_device(model):
    return model.get_input_embeddings().weight.device


def get_layer_device(model, layer_idx: int):
    block = model.get_submodule(f"model.layers.{layer_idx}")
    try:
        return next(block.parameters()).device
    except StopIteration:
        return get_model_input_device(model)


MODEL_INPUT_DEVICE = get_model_input_device(model)
TARGET_LAYER_DEVICE = get_layer_device(model, TARGET_LAYER)

print("REQUESTED_MODEL_NAME:", REQUESTED_MODEL_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_SHORT_NAME:", MODEL_SHORT_NAME)
print("TARGET_LAYER:", TARGET_LAYER)
print("MODEL_INPUT_DEVICE:", MODEL_INPUT_DEVICE)
print("TARGET_LAYER_DEVICE:", TARGET_LAYER_DEVICE)
print("SAE_BACKEND:", SAE_BACKEND)
print("SAE_REPO_ID:", SAE_REPO_ID)
print("SAE_FILENAME:", SAE_FILENAME)
print("SAE_HYPERPARAMS_FILENAME:", SAE_HYPERPARAMS_FILENAME)
print("PROMPT_FORMAT:", PROMPT_FORMAT)
print("BLOCK_END_TOKEN:", BLOCK_END_TOKEN)

def _as_tensor(value) -> torch.Tensor:
    if isinstance(value, np.ndarray):
        value = torch.from_numpy(value)
    if torch.is_tensor(value):
        return value.detach()
    return torch.tensor(value)


def _expand_threshold(threshold: torch.Tensor, n_features: int) -> torch.Tensor:
    threshold = _as_tensor(threshold).reshape(-1)
    if threshold.numel() == 1:
        threshold = threshold.repeat(int(n_features))
    if threshold.numel() != int(n_features):
        raise ValueError(
            f"Threshold has {threshold.numel()} values; expected 1 or {n_features}."
        )
    return threshold.contiguous()

class LlamaScopeResidualSAE(torch.nn.Module):
    """Direct inference wrapper for OpenMOSS Llama Scope residual SAEs."""

    def __init__(self, state_dict, hyperparams, device, dtype):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype
        self.hyperparams = dict(hyperparams)

        self.d_model = int(hyperparams["d_model"])
        self.n_features = int(hyperparams["d_sae"])
        self.k = int(hyperparams.get("top_k", 0) or 0) or None
        self.act_fn = str(hyperparams.get("act_fn", "jumprelu")).lower()
        self.sparsity_include_decoder_norm = bool(
            hyperparams.get("sparsity_include_decoder_norm", False)
        )
        self.apply_decoder_bias_to_pre_encoder = bool(
            hyperparams.get("apply_decoder_bias_to_pre_encoder", False)
        )

        def optional_tensor(candidate_keys, shape_options=None):
            for key in candidate_keys:
                if key in state_dict:
                    tensor = _as_tensor(state_dict[key]).cpu()
                    if shape_options is None or tuple(tensor.shape) in {
                        tuple(x) for x in shape_options
                    }:
                        return tensor
            for key, value in state_dict.items():
                if not torch.is_tensor(value):
                    continue
                tensor = value.detach().cpu()
                if shape_options is not None and tuple(tensor.shape) not in {
                    tuple(x) for x in shape_options
                }:
                    continue
                lowered = key.lower()
                if any(candidate.lower() in lowered for candidate in candidate_keys):
                    print(f"Using Llama Scope fuzzy key match: {key}")
                    return tensor
            return None

        W_enc = optional_tensor(
            ["W_E", "encoder.weight", "W_enc"],
            [(self.d_model, self.n_features), (self.n_features, self.d_model)],
        )
        W_dec = optional_tensor(
            ["W_D", "decoder.weight", "W_dec"],
            [(self.n_features, self.d_model), (self.d_model, self.n_features)],
        )
        if W_enc is None or W_dec is None:
            available = {
                key: tuple(value.shape)
                for key, value in state_dict.items()
                if torch.is_tensor(value)
            }
            raise KeyError(f"Could not locate Llama Scope weights. Available: {available}")
        if W_enc.shape == (self.n_features, self.d_model):
            W_enc = W_enc.T.contiguous()
        if W_dec.shape == (self.d_model, self.n_features):
            W_dec = W_dec.T.contiguous()

        b_enc = optional_tensor(
            ["b_E", "encoder.bias", "b_enc"], [(self.n_features,)]
        )
        b_dec = optional_tensor(
            ["b_D", "decoder.bias", "b_dec"], [(self.d_model,)]
        )
        if b_enc is None:
            b_enc = torch.zeros(self.n_features)
        if b_dec is None or not bool(hyperparams.get("use_decoder_bias", True)):
            b_dec = torch.zeros(self.d_model)

        log_threshold = optional_tensor(
            ["activation_function.log_jumprelu_threshold", "log_jumprelu_threshold"],
            [(self.n_features,), (1,)],
        )
        if log_threshold is not None:
            threshold = log_threshold.float().exp()
        else:
            threshold = optional_tensor(
                [
                    "activation_function.jumprelu_threshold",
                    "jumprelu_threshold",
                    "threshold",
                ],
                [(self.n_features,), (1,)],
            )
        if threshold is None:
            threshold = torch.tensor(float(hyperparams.get("jump_relu_threshold", 0.0)))
        threshold = _expand_threshold(threshold, self.n_features)

        # Match Llamascopium's inference-time parameter standardization for
        # dataset-wise activation normalization.
        self.norm_activation_standardized = False
        if (
            hyperparams.get("norm_activation") == "dataset-wise"
            and hyperparams.get("dataset_average_activation_norm") is not None
        ):
            average_norm = hyperparams["dataset_average_activation_norm"]
            hook_in = str(hyperparams.get("hook_point_in", "in"))
            hook_out = str(hyperparams.get("hook_point_out", hook_in))
            norm_in = float(average_norm.get(hook_in, average_norm.get("in")))
            norm_out = float(average_norm.get(hook_out, average_norm.get("out", norm_in)))
            input_norm_factor = math.sqrt(self.d_model) / norm_in
            output_norm_factor = math.sqrt(self.d_model) / norm_out
            b_enc = b_enc / input_norm_factor
            b_dec = b_dec / output_norm_factor
            W_dec = W_dec * (input_norm_factor / output_norm_factor)
            self.norm_activation_standardized = True

        decoder_norm = torch.linalg.vector_norm(W_dec.float(), dim=1).clamp_min(1e-8)

        self.register_buffer("W_enc", W_enc.to(self.device, self.dtype))
        self.register_buffer("W_dec", W_dec.to(self.device, self.dtype))
        self.register_buffer("b_enc", b_enc.reshape(-1).to(self.device, self.dtype))
        self.register_buffer("b_dec", b_dec.reshape(-1).to(self.device, self.dtype))
        self.register_buffer("threshold", threshold.to(self.device, self.dtype))
        self.register_buffer("decoder_norm", decoder_norm.to(self.device, self.dtype))

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        if self.apply_decoder_bias_to_pre_encoder:
            x = x - self.b_dec
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        pre = self.process_sae_in(x) @ self.W_enc + self.b_enc
        if self.sparsity_include_decoder_norm:
            pre = pre * self.decoder_norm
        return pre

    def feature_fire_mask(self, pre: torch.Tensor) -> torch.Tensor:
        if self.act_fn == "jumprelu":
            return pre > self.threshold
        if self.act_fn == "relu":
            return pre > 0
        if self.act_fn == "topk":
            return self.activation_fn(pre) != 0
        raise ValueError(f"Unsupported Llama Scope act_fn={self.act_fn!r}")

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        if self.act_fn == "jumprelu":
            acts = torch.where(pre > self.threshold, pre, torch.zeros_like(pre))
        elif self.act_fn == "relu":
            acts = torch.relu(pre)
        elif self.act_fn == "topk":
            if self.k is None or self.k <= 0:
                raise ValueError("Llama Scope TopK checkpoint has no positive k.")
            values, indices = torch.topk(
                pre, k=min(int(self.k), pre.shape[-1]), dim=-1
            )
            values = torch.relu(values)
            acts = torch.zeros_like(pre).scatter(-1, indices, values)
        else:
            raise ValueError(f"Unsupported Llama Scope act_fn={self.act_fn!r}")
        if self.sparsity_include_decoder_norm:
            acts = acts / self.decoder_norm
        return acts

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec

class QwenBatchTopKSAE(torch.nn.Module):
    """
    Minimal inference wrapper for Adam Karvonen's Qwen3 BatchTopK SAEs.

    The checkpoint has dictionary_learning-style keys:
      b_dec, k, threshold, decoder.weight, encoder.weight, encoder.bias

    This wrapper exposes an SAE-Lens-like interface:
      W_enc: [d_model, n_features]
      W_dec: [n_features, d_model]
      b_enc: [n_features]
      b_dec: [d_model]
      process_sae_in(x)
      activation_fn(pre)
      encode(x)
      decode(acts)
    """

    def __init__(self, state_dict, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        if isinstance(state_dict, dict) and "model" in state_dict and isinstance(state_dict["model"], dict):
            state_dict = state_dict["model"]

        b_dec = state_dict["b_dec"].detach()
        b_enc = state_dict.get("encoder.bias", state_dict.get("b_enc")).detach()
        enc_w = state_dict.get("encoder.weight", state_dict.get("W_enc")).detach()
        dec_w = state_dict.get("decoder.weight", state_dict.get("W_dec")).detach()
        threshold = state_dict["threshold"]
        threshold = threshold.detach() if torch.is_tensor(threshold) else torch.tensor(threshold)
        k = state_dict.get("k", None)
        if torch.is_tensor(k):
            k = int(k.item())

        d_model = int(b_dec.numel())
        n_features = int(b_enc.numel())

        if enc_w.shape == (n_features, d_model):
            W_enc = enc_w.T.contiguous()
        elif enc_w.shape == (d_model, n_features):
            W_enc = enc_w.contiguous()
        else:
            raise ValueError(
                f"Unexpected encoder.weight shape {tuple(enc_w.shape)} "
                f"for d_model={d_model}, n_features={n_features}"
            )

        if dec_w.shape == (d_model, n_features):
            W_dec = dec_w.T.contiguous()
        elif dec_w.shape == (n_features, d_model):
            W_dec = dec_w.contiguous()
        else:
            raise ValueError(
                f"Unexpected decoder.weight shape {tuple(dec_w.shape)} "
                f"for d_model={d_model}, n_features={n_features}"
            )

        self.register_buffer("W_enc", W_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.to(device=self.device, dtype=self.dtype))

        self.k = k
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        return x - self.b_dec

    def process_sae_out(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, pre, torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec

class GemmaScopeJumpReLUSAE(torch.nn.Module):
    """
    Minimal inference wrapper for Gemma Scope JumpReLU residual-stream SAEs.
    """

    def __init__(self, params, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        def get_param(name, *aliases):
            for key in (name,) + aliases:
                if key in params:
                    value = params[key]
                    if isinstance(value, np.ndarray):
                        value = torch.from_numpy(value)
                    return value.detach() if torch.is_tensor(value) else torch.tensor(value)
            raise KeyError(f"Missing SAE parameter {name!r}. Available keys: {sorted(params.keys())}")

        W_enc = get_param("W_enc")
        W_dec = get_param("W_dec")
        b_enc = get_param("b_enc")
        b_dec = get_param("b_dec")
        threshold = get_param("threshold", "thresholds")

        if W_enc.ndim != 2 or W_dec.ndim != 2:
            raise ValueError(f"Expected 2D W_enc/W_dec, got {tuple(W_enc.shape)} and {tuple(W_dec.shape)}")

        d_model = int(W_enc.shape[0])
        n_features = int(W_enc.shape[1])

        if W_dec.shape == (d_model, n_features):
            W_dec = W_dec.T.contiguous()
        if W_dec.shape != (n_features, d_model):
            raise ValueError(f"Expected W_dec shape {(n_features, d_model)}, got {tuple(W_dec.shape)}")
        if b_enc.numel() != n_features:
            raise ValueError(f"Expected b_enc length {n_features}, got {b_enc.numel()}")
        if b_dec.numel() != d_model:
            raise ValueError(f"Expected b_dec length {d_model}, got {b_dec.numel()}")
        if threshold.numel() != n_features:
            raise ValueError(f"Expected threshold length {n_features}, got {threshold.numel()}")

        self.register_buffer("W_enc", W_enc.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))

        self.k = None
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        # Gemma Scope folds the pre-encoder bias into the encoder parameters.
        return x

    def process_sae_out(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, torch.relu(pre), torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec

def load_qwen_batchtopk_sae(repo_id: str, filename: str, device=DEVICE, dtype=DTYPE):
    print("Downloading/loading Qwen BatchTopK SAE checkpoint:", repo_id, filename)
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_files_only=SAE_LOCAL_FILES_ONLY,
        token=False,
    )
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(path, map_location="cpu")
    sae_obj = QwenBatchTopKSAE(state, device=device, dtype=dtype)
    print(
        f"Loaded SAE: d_model={sae_obj.d_model}, n_features={sae_obj.n_features}, "
        f"k={sae_obj.k}, threshold={float(sae_obj.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae_obj


def load_gemma_scope_jumprelu_sae(repo_id: str, filename: str, device=DEVICE, dtype=DTYPE):
    print("Downloading/loading Gemma Scope SAE checkpoint:", repo_id, filename)
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_files_only=SAE_LOCAL_FILES_ONLY,
        token=False,
    )
    with np.load(path, allow_pickle=False) as params_np:
        params = {k: torch.from_numpy(params_np[k]) for k in params_np.files}
    sae_obj = GemmaScopeJumpReLUSAE(params, device=device, dtype=dtype)
    print(
        f"Loaded SAE: d_model={sae_obj.d_model}, n_features={sae_obj.n_features}, "
        f"mean_threshold={float(sae_obj.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae_obj


def load_llama_scope_direct_sae(
    repo_id: str,
    filename: str,
    hyperparams_filename: str,
    device=DEVICE,
    dtype=DTYPE,
):
    try:
        from safetensors.torch import load_file as load_safetensors_file
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Install safetensors to load Llama Scope: pip install safetensors"
        ) from exc

    print("Downloading/loading Llama Scope SAE checkpoint:", repo_id, filename)
    checkpoint_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_files_only=SAE_LOCAL_FILES_ONLY,
        token=False,
    )
    hyperparams_path = hf_hub_download(
        repo_id=repo_id,
        filename=hyperparams_filename,
        local_files_only=SAE_LOCAL_FILES_ONLY,
        token=False,
    )
    with open(hyperparams_path, "r", encoding="utf-8") as handle:
        hyperparams = json.load(handle)
    state = load_safetensors_file(checkpoint_path, device="cpu")
    sae_obj = LlamaScopeResidualSAE(
        state,
        hyperparams=hyperparams,
        device=device,
        dtype=dtype,
    )
    print(
        f"Loaded SAE: d_model={sae_obj.d_model}, n_features={sae_obj.n_features}, "
        f"act_fn={sae_obj.act_fn}, k={sae_obj.k}"
    )
    return sae_obj


def load_configured_sae():
    if SAE_BACKEND == "qwen_batchtopk":
        return load_qwen_batchtopk_sae(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            device=TARGET_LAYER_DEVICE,
            dtype=DTYPE,
        )
    if SAE_BACKEND == "gemma_scope_npz":
        return load_gemma_scope_jumprelu_sae(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            device=TARGET_LAYER_DEVICE,
            dtype=DTYPE,
        )
    if SAE_BACKEND == "llama_scope_direct":
        return load_llama_scope_direct_sae(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            hyperparams_filename=SAE_HYPERPARAMS_FILENAME,
            device=TARGET_LAYER_DEVICE,
            dtype=DTYPE,
        )
    raise ValueError(f"Unknown SAE_BACKEND={SAE_BACKEND!r}")


sae = load_configured_sae()
sae.eval()

_model_cfg = getattr(model.config, "text_config", model.config)
_model_hidden_size = int(getattr(_model_cfg, "hidden_size"))
_model_num_layers = int(getattr(_model_cfg, "num_hidden_layers"))
if not 0 <= int(TARGET_LAYER) < _model_num_layers:
    raise ValueError(
        f"TARGET_LAYER={TARGET_LAYER} outside [0, {_model_num_layers})."
    )
if _model_num_layers != int(EXPECTED_MODEL_NUM_LAYERS):
    raise ValueError(
        f"Expected {EXPECTED_MODEL_NUM_LAYERS} transformer layers, found {_model_num_layers}."
    )
if _model_hidden_size != int(EXPECTED_D_MODEL):
    raise ValueError(
        f"Expected model hidden size {EXPECTED_D_MODEL}, found {_model_hidden_size}."
    )
if int(sae.d_model) != int(EXPECTED_D_MODEL):
    raise ValueError(
        f"SAE/model width mismatch: sae.d_model={sae.d_model}, expected={EXPECTED_D_MODEL}."
    )
if int(sae.n_features) != int(EXPECTED_D_SAE):
    raise ValueError(
        f"Expected {EXPECTED_D_SAE} SAE features, found {sae.n_features}."
    )
print("Model/SAE compatibility checks passed.")

# ============================================================
# 2. SAE and model utilities
# ============================================================

def clear_all_hooks(model: torch.nn.Module):
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()


@torch.no_grad()
def sae_logits_before_jumprelu(
    sae,
    x: torch.Tensor,
    filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
) -> torch.Tensor:
    """
    Return SAE encoder pre-activations before thresholding.

    x shape can be (..., d_model). Output shape is (..., n_features).
    """
    x_sae = x.to(device=sae.device, dtype=sae.dtype)
    hidden_pre = sae.encode_pre(x_sae)

    if filter_high_norm_tokens and x_sae.ndim >= 3:
        norms = x_sae.detach().float().norm(dim=-1)  # [B, S]
        med = norms.median(dim=1, keepdim=True).values.clamp_min(1e-6)
        high_norm_mask = norms > (HIGH_NORM_FILTER_MULTIPLIER * med)
        hidden_pre = hidden_pre.masked_fill(high_norm_mask.to(hidden_pre.device)[..., None], 0.0)

    return hidden_pre


@torch.no_grad()
def sae_activations(
    sae,
    x: torch.Tensor,
    filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
) -> torch.Tensor:
    pre = sae_logits_before_jumprelu(
        sae=sae,
        x=x,
        filter_high_norm_tokens=filter_high_norm_tokens,
    )
    return sae.activation_fn(pre)


@torch.no_grad()
def get_layer_output(model, input_ids, attention_mask, layer_idx: int):
    """
    Run a forward pass and capture the output tensor of model.layers[layer_idx].
    Returns shape [batch, seq, hidden].
    """
    captured = {}

    def hook_fn(module, inputs, output):
        captured["x"] = output[0].detach() if isinstance(output, tuple) else output.detach()
        return output

    block = model.get_submodule(f"model.layers.{layer_idx}")
    handle = block.register_forward_hook(hook_fn)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        handle.remove()

    if "x" not in captured:
        raise RuntimeError(f"Failed to capture output from layer {layer_idx}")

    return captured["x"]

print({
    "task": PROMPT_DATASET_NAME,
    "model": MODEL_NAME,
    "target_layer": TARGET_LAYER,
    "sae_backend": SAE_BACKEND,
    "sae_repo": SAE_REPO_ID,
    "sae_file": SAE_FILENAME,
    "prompt_format": PROMPT_FORMAT,
    "block_score_mode": BLOCK_SCORE_MODE,
    "block_token_offsets_from_end": BLOCK_TOKEN_OFFSETS_FROM_END,
    "high_norm_filter": FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
})

In [ ]:

# ============================================================
# 3. BFCL Tool Calling 8-block LFH prompt generation
# ============================================================

from typing import Any, Dict, List, Optional, Tuple, Sequence, Union

prompt_template_agnews = None  # Backward-compatible placeholder.
train_data_agnews = None       # Backward-compatible placeholder.


def normalize_text(x: Any, max_chars: Optional[int] = None) -> str:
    s = str(x).replace("\n", " ").replace("\t", " ").strip()
    s = re.sub(r"\s+", " ", s)
    if max_chars is not None and len(s) > max_chars:
        # Do not append artificial ellipses before block headers; repeated "..."
        # creates an unintended boundary cue for LFH/DPE analyses.
        truncated = s[:max_chars].rstrip()
        last_space = truncated.rfind(" ")
        if last_space >= int(0.55 * max_chars):
            truncated = truncated[:last_space].rstrip()
        s = truncated
    return s


BFCL_REPO_ID = "gorilla-llm/Berkeley-Function-Calling-Leaderboard"
BFCL_DATA_FILE = "BFCL_v3_multiple.json"
MAX_TOOL_DESCRIPTION_CHARS = 260
MAX_PARAM_DESCRIPTION_CHARS = 120
MAX_PARAMS_PER_TOOL = 8
_TOOL_POOL_CACHE = None


def load_json_or_jsonl_from_hf(repo_id: str, filename: str, repo_type: str = "dataset"):
    """Download and parse a JSON or JSONL file from Hugging Face without relying on dataset scripts."""
    path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type=repo_type, token=False)
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            for key in ["data", "examples", "rows"]:
                if key in obj and isinstance(obj[key], list):
                    return obj[key]
            return [obj]
    except json.JSONDecodeError:
        pass
    rows = []
    for line_no, line in enumerate(text.splitlines(), start=1):
        s = line.strip()
        if not s:
            continue
        try:
            rows.append(json.loads(s))
        except json.JSONDecodeError as e:
            raise ValueError(f"Failed to parse {filename} line {line_no}: {e}\n{s[:500]}")
    return rows


def _canonical_json(x: Any) -> str:
    try:
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    except TypeError:
        return str(x)


def compact_tool_schema(func: Dict[str, Any], idx: int) -> str:
    name = normalize_text(func.get("name", f"tool_{idx}"))
    desc = normalize_text(func.get("description", ""), max_chars=MAX_TOOL_DESCRIPTION_CHARS)
    params = func.get("parameters", {}) or {}
    props = params.get("properties", {}) if isinstance(params, dict) else {}
    required = params.get("required", []) if isinstance(params, dict) else []
    param_parts = []
    if isinstance(props, dict):
        for p_name, p_info in list(props.items())[:MAX_PARAMS_PER_TOOL]:
            if isinstance(p_info, dict):
                p_type = p_info.get("type", "unknown")
                p_desc = normalize_text(p_info.get("description", ""), max_chars=MAX_PARAM_DESCRIPTION_CHARS)
            else:
                p_type = "unknown"
                p_desc = ""
            req_mark = "required" if p_name in required else "optional"
            if p_desc:
                param_parts.append(f"{p_name}: {p_type} ({req_mark}; {p_desc})")
            else:
                param_parts.append(f"{p_name}: {p_type} ({req_mark})")
    params_str = "; ".join(param_parts) if param_parts else "none"
    return f"Tool {idx}:\nName: {name}\nDescription: {desc}\nParameters: {params_str}"


def build_tool_pool():
    global _TOOL_POOL_CACHE
    if _TOOL_POOL_CACHE is not None:
        return _TOOL_POOL_CACHE
    print("Loading BFCL tool-use data:", BFCL_REPO_ID, BFCL_DATA_FILE)
    data_rows = load_json_or_jsonl_from_hf(BFCL_REPO_ID, BFCL_DATA_FILE, repo_type="dataset")
    tool_pool = []
    seen = set()
    for row_idx, row in enumerate(data_rows):
        funcs = row.get("function", []) or []
        if not isinstance(funcs, list):
            continue
        for func_idx, func in enumerate(funcs):
            if not isinstance(func, dict):
                continue
            name = normalize_text(func.get("name", f"tool_{row_idx}_{func_idx}"))
            if not name:
                continue
            sig = _canonical_json({"name": func.get("name", ""), "description": func.get("description", ""), "parameters": func.get("parameters", {})})
            if sig in seen:
                continue
            seen.add(sig)
            tool_pool.append({"tool_idx": len(tool_pool), "source_row_idx": row_idx, "source_func_idx": func_idx, "name": name, "schema": func, "signature": sig})
    if len(tool_pool) < max(NUM_BLOCKS_TOTAL + NUM_PREVIOUS + NUM_TEST_QUERY, 100):
        print(f"Warning: tool pool has {len(tool_pool)} unique tools; sampling will reuse across target queries if needed.")
    _TOOL_POOL_CACHE = tool_pool
    print(f"Loaded {len(tool_pool)} unique individual tools from BFCL.")
    return tool_pool


def render_item_prompt_body(items: List[Dict[str, Any]], query_row: Any = None) -> str:
    header = (
        "You are given a catalog of available tools/functions. Each numbered item is one tool definition.\n"
        "Study the list of tools and their parameter schemas. Do not answer; just track the sequence of tool definitions.\n\n"
        "Available tool definitions:\n"
    )
    blocks = [compact_tool_schema(t["schema"], i + 1) for i, t in enumerate(items)]
    return header + "\n\n".join(blocks) + "\n"


def get_item_pool_and_query_pool(num_test_query: int, num_previous: int):
    tool_pool = build_tool_pool()
    return tool_pool, [None] * int(num_test_query)



def ensure_single_bos_prefix(text: str, tokenizer) -> str:
    bos = globals().get("PROMPT_BOS_TOKEN", "") or ""
    if not bos:
        return text
    if text.startswith(bos):
        return text
    return bos + text


def _manual_qwen_chatml_render(system_content: str, user_content: str) -> str:
    return (
        f"<|im_start|>system\n{system_content}<|im_end|>\n"
        f"<|im_start|>user\n{user_content}<|im_end|>\n"
    )


def render_model_prompt(tokenizer, user_content: str, system_content: Optional[str] = None):
    """Render base models as plain text and chat models as a single native user turn."""
    system_content = system_content if system_content is not None else SYSTEM_PROMPT
    prompt_format = globals().get("PROMPT_FORMAT", "raw")

    # Base/raw models: no chat template and no chat tokens.
    if prompt_format == "raw":
        plain = f"{system_content}\n\n{user_content}" if system_content else user_content
        return ensure_single_bos_prefix(plain, tokenizer)

    # Instruction/chat models: one user message containing the whole ordered list.
    rendered_user_content = user_content
    messages = []
    if prompt_format == "gemma_chat_single_user":
        # Gemma chat has no system role; fold the instruction into the user message.
        rendered_user_content = f"{system_content}\n\n{user_content}" if system_content else user_content
        messages = [{"role": "user", "content": rendered_user_content}]
    elif prompt_format in {"llama_chat_single_user", "qwen_chat_single_user"}:
        if system_content:
            messages.append({"role": "system", "content": system_content})
        messages.append({"role": "user", "content": rendered_user_content})
    else:
        raise ValueError(f"Unknown PROMPT_FORMAT={prompt_format!r}")

    try:
        rendered = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            **globals().get("CHAT_TEMPLATE_KWARGS", {}),
        )
        return ensure_single_bos_prefix(rendered, tokenizer)
    except TypeError:
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return ensure_single_bos_prefix(rendered, tokenizer)
    except Exception as e:
        print("Warning: tokenizer.apply_chat_template failed. Falling back to a manual/simple prompt.")
        print("Reason:", repr(e))
        if prompt_format == "qwen_chat_single_user":
            return ensure_single_bos_prefix(_manual_qwen_chatml_render(system_content, rendered_user_content), tokenizer)
        return ensure_single_bos_prefix(f"{system_content}\n\n{user_content}", tokenizer)


def _sample_one_excluding(rng: random.Random, population_size: int, forbidden: set):
    if population_size <= len(forbidden):
        raise ValueError("Population too small for exclusion-based sampling.")
    while True:
        idx = rng.randrange(population_size)
        if idx not in forbidden:
            return idx


def build_prompts_all_agnews(
    train_data_agnews=None,
    prompt_template_agnews=None,
    num_test_query=25,
    num_previous=25,
    varied_block_num=None,
    target_block_num=None,
    test_query_indices=None,
):
    """
    Generic LFH prompt builder for ordered Tool/Document blocks.

    For each fixed target/query index:
      - the target semantic block is fixed at block 8;
      - all non-source context blocks are fixed;
      - the configured source block is resampled across variants.
    The returned variable names keep the AGNews suffix for compatibility with
    downstream LFH analysis cells.
    """
    varied_block_num = int(LFH_SOURCE_BLOCK_NUM if varied_block_num is None else varied_block_num)
    target_block_num = int(LFH_TARGET_BLOCK_NUM if target_block_num is None else target_block_num)
    if target_block_num != NUM_BLOCKS_TOTAL:
        raise NotImplementedError(f"This template fills block {NUM_BLOCKS_TOTAL} as the fixed target; got target_block_num={target_block_num}.")
    if not (1 <= varied_block_num < target_block_num):
        raise ValueError(f"varied_block_num must be before target block {target_block_num}, got {varied_block_num}.")

    item_pool, query_pool = get_item_pool_and_query_pool(num_test_query=int(num_test_query), num_previous=int(num_previous))
    n_items_total = len(item_pool)
    source_pos = block_num_to_index(varied_block_num, total_blocks=NUM_BLOCKS_TOTAL)
    num_context_blocks = NUM_BLOCKS_TOTAL - 1

    prompts_all_agnews = {i: [] for i in range(num_test_query)}
    prompt_bodies_all_agnews = {i: [] for i in range(num_test_query)}
    varied_source_indices_by_query = {}

    if test_query_indices is None:
        if TASK_TYPE == "nq_document":
            test_query_indices = list(range(int(num_test_query)))
        else:
            rng = random.Random(42)
            test_query_indices = rng.sample(range(n_items_total), int(num_test_query))
    else:
        test_query_indices = list(test_query_indices)[:int(num_test_query)]

    for i in range(int(num_test_query)):
        if i % 10 == 0:
            print(
                f"building {PROMPT_DATASET_NAME} prompts for query {i}; "
                f"varied source block={varied_block_num}, fixed target block={target_block_num}"
            )

        rng = random.Random(10_000 + i)
        query_row = query_pool[test_query_indices[i] % len(query_pool)] if TASK_TYPE == "nq_document" else None

        # Select the fixed target item for block 8.
        if TASK_TYPE == "bfcl_tool":
            target_item_idx = int(test_query_indices[i]) % n_items_total
        else:
            target_item_idx = _sample_one_excluding(rng, n_items_total, forbidden=set())

        # Context blocks 1..7. Keep all non-source context blocks fixed.
        forbidden = {target_item_idx}
        context_indices = []
        while len(context_indices) < num_context_blocks:
            idx = _sample_one_excluding(rng, n_items_total, forbidden | set(context_indices))
            context_indices.append(idx)

        fixed_context_indices = {idx for pos, idx in enumerate(context_indices) if pos != source_pos}
        used_varied_source_indices = set()
        varied_source_indices_by_query[i] = []

        for k in range(int(num_previous)):
            if k == 0:
                source_idx = context_indices[source_pos]
            else:
                source_idx = _sample_one_excluding(
                    rng,
                    n_items_total,
                    forbidden | fixed_context_indices | used_varied_source_indices,
                )
                context_indices[source_pos] = source_idx

            used_varied_source_indices.add(source_idx)
            varied_source_indices_by_query[i].append(source_idx)

            item_indices = list(context_indices) + [target_item_idx]
            items = [item_pool[idx] for idx in item_indices]
            prompt_body = render_item_prompt_body(items, query_row=query_row)
            prompt = render_model_prompt(tokenizer=tokenizer, user_content=prompt_body, system_content=SYSTEM_PROMPT)

            prompt_bodies_all_agnews[i].append(prompt_body)
            prompts_all_agnews[i].append(prompt)

    return prompts_all_agnews, prompt_bodies_all_agnews, test_query_indices, varied_source_indices_by_query


prompts_all_agnews, prompt_bodies_all_agnews, test_query_indices, varied_source_indices_by_query = build_prompts_all_agnews(
    train_data_agnews=train_data_agnews,
    prompt_template_agnews=prompt_template_agnews,
    num_test_query=NUM_TEST_QUERY,
    num_previous=NUM_PREVIOUS,
    varied_block_num=LFH_SOURCE_BLOCK_NUM,
    target_block_num=LFH_TARGET_BLOCK_NUM,
)

# Backward-compatible alias for downstream analysis.
varied_previous_indices_by_query = varied_source_indices_by_query

assert len(prompts_all_agnews) == NUM_TEST_QUERY
assert len(prompts_all_agnews[0]) == NUM_PREVIOUS

print("MODEL_SHORT_NAME:", MODEL_SHORT_NAME)
print("PROMPT_DATASET_NAME:", PROMPT_DATASET_NAME)
print("PROMPT_FORMAT:", PROMPT_FORMAT)
print("LFH source block:", LFH_SOURCE_BLOCK_NUM)
print("LFH target block:", LFH_TARGET_BLOCK_NUM)
print("Example rendered prompt head:")
print(prompts_all_agnews[0][0][:1600])

In [ ]:

# ============================================================
# 4. Block token-selection helpers with selectable scoring
# ============================================================

# Matches each semantic-block header. For these variants, blocks are headed as "Tool N:".
EXAMPLE_HEADER_RE = re.compile(
    rf"(?m)^{re.escape(ITEM_WORD)}\s+\d+:?\s*\r?\n",
    flags=re.IGNORECASE,
)


class TokenSelection:
    """
    Token selection for one exemplar block.

    The object behaves like a legacy (start, end) span when unpacked, so older
    plotting/logging code can still do ``s, e = selection``. Actual analyses use
    ``selection.indices`` through token_selection_to_* helpers, which allows
    non-contiguous choices such as [-1, -3] if desired.
    """

    def __init__(self, indices, *, full_start=None, full_end=None, label=None):
        indices = tuple(sorted(dict.fromkeys(int(i) for i in indices)))
        if len(indices) == 0:
            raise ValueError("TokenSelection requires at least one token index.")
        self.indices = indices
        self.start = int(indices[0])
        self.end = int(indices[-1]) + 1
        self.full_start = int(full_start) if full_start is not None else self.start
        self.full_end = int(full_end) if full_end is not None else self.end
        self.label = label

    def __iter__(self):
        # Backward-compatible unpacking: s, e = selection
        yield self.start
        yield self.end

    def __getitem__(self, item):
        return (self.start, self.end)[item]

    def __len__(self):
        # Legacy span-like length. Use n_selected for number of selected tokens.
        return 2

    @property
    def n_selected(self):
        return len(self.indices)

    @property
    def is_contiguous(self):
        return self.n_selected == (self.end - self.start)

    def __repr__(self):
        idx = list(self.indices)
        if len(idx) > 10:
            idx_repr = f"{idx[:5]} ... {idx[-5:]}"
        else:
            idx_repr = repr(idx)
        return (
            f"TokenSelection(start={self.start}, end={self.end}, "
            f"n_selected={self.n_selected}, contiguous={self.is_contiguous}, "
            f"indices={idx_repr})"
        )


def token_selection_to_indices(selection):
    """Return selected token indices as a Python list."""
    if isinstance(selection, TokenSelection):
        return list(selection.indices)
    s, e = selection
    return list(range(int(s), int(e)))


def token_selection_to_torch_indices(selection, device=None):
    """Return selected token indices as a torch.LongTensor."""
    return torch.tensor(token_selection_to_indices(selection), dtype=torch.long, device=device)


def token_selection_bounds(selection):
    """Return the legacy [start, end) envelope for logging/plotting."""
    if isinstance(selection, TokenSelection):
        return int(selection.start), int(selection.end)
    s, e = selection
    return int(s), int(e)


def token_selection_mean(x: torch.Tensor, selection, dim: int = 0):
    """Mean over selected token positions along ``dim``."""
    idx = token_selection_to_torch_indices(selection, device=x.device)
    return x.index_select(dim, idx).mean(dim=dim)


def _normalize_block_score_mode(mode):
    """Normalize user-facing BLOCK_SCORE_MODE strings."""
    mode_str = str(mode).strip()
    mode_key = mode_str.lower().replace("-", "_").replace(" ", "_")

    # User-facing full-exemplar mode. "content" is kept as a backward-compatible alias.
    if mode_key in {"all", "all_tokens", "full", "full_block", "content"}:
        return "all"

    # New exact relative-position mode.
    if mode_key in {
        "selected_offsets_from_end",
        "select_offsets_from_end",
        "token_offsets_from_end",
        "selected_token_offsets_from_end",
        "selected_tokens_from_end",
        "selected_tokens",
        "offsets_from_end",
    }:
        return "selected_offsets_from_end"

    # Legacy last-K family of modes, including "last_k_with_end_of_turn".
    if mode_key.startswith("last_k"):
        return "last_k"

    raise ValueError(
        f"Unknown BLOCK_SCORE_MODE={mode!r}. Supported modes are 'selected_offsets_from_end', 'All', and legacy last_k* modes."
    )


def _normalize_offsets_from_end(offsets):
    """
    Validate exact token offsets from the end of each exemplar span.

    Negative offsets count backward from the exemplar end:
      -1 = final token, -2 = second-to-final token, etc.
    """
    if offsets is None:
        return None
    if isinstance(offsets, (int, np.integer)):
        offsets = [int(offsets)]
    offsets = [int(x) for x in list(offsets)]
    if len(offsets) == 0:
        raise ValueError("BLOCK_TOKEN_OFFSETS_FROM_END must be non-empty when using selected-offset mode.")
    if any(x >= 0 for x in offsets):
        raise ValueError(
            "BLOCK_TOKEN_OFFSETS_FROM_END expects negative integers only: "
            "-1 is the final token, -2 is the second-to-final token, etc."
        )
    return offsets


def _legacy_last_k_to_offsets(last_k_tokens: int):
    last_k_tokens = int(last_k_tokens)
    if last_k_tokens <= 0:
        raise ValueError(f"BLOCK_LAST_K_TOKENS must be positive for legacy last-K modes, got {last_k_tokens}.")
    return [-i for i in range(1, last_k_tokens + 1)]


def _get_block_scoring_config():
    """Read scoring options, with safe defaults for older cached notebooks."""
    raw_mode = globals().get("BLOCK_SCORE_MODE", "selected_offsets_from_end")
    mode = _normalize_block_score_mode(raw_mode)
    end_token = globals().get("BLOCK_END_TOKEN", None)

    if mode == "all":
        return mode, end_token, None

    if mode == "last_k":
        # Strict backward compatibility: legacy last_k* modes are controlled by
        # BLOCK_LAST_K_TOKENS, irrespective of BLOCK_TOKEN_OFFSETS_FROM_END.
        offsets = _legacy_last_k_to_offsets(globals().get("BLOCK_LAST_K_TOKENS", 8))
    else:
        explicit_offsets = globals().get("BLOCK_TOKEN_OFFSETS_FROM_END", None)
        if explicit_offsets is not None:
            offsets = _normalize_offsets_from_end(explicit_offsets)
        else:
            # Safe fallback for notebooks configured before BLOCK_TOKEN_OFFSETS_FROM_END existed.
            offsets = _legacy_last_k_to_offsets(globals().get("BLOCK_LAST_K_TOKENS", 8))

    if len(offsets) == 0:
        raise ValueError("No token offsets selected for block scoring.")

    return "selected_offsets_from_end", end_token, offsets


def _trim_trailing_chat_tail(prompt: str, start: int, end: int) -> int:
    """Exclude final chat-template terminators / assistant-generation prompts from the last semantic block."""
    tail_markers = [
        "<end_of_turn>",
        "<|eot_id|>",
        "<|im_end|>",
        "<|start_header_id|>assistant<|end_header_id|>",
        "<|im_start|>assistant",
    ]
    candidates = []
    for marker in tail_markers:
        pos = prompt.find(marker, start, end)
        if pos >= 0:
            candidates.append(pos)
    if candidates:
        end = min(candidates)
    return end


def find_example_block_char_spans(prompt: str, expected_num_blocks: int):
    """
    Find character spans of all semantic blocks headed by ITEM_WORD, e.g. Tool 1 or Document 1.

    For list-style BFCL/NQ prompts, chat-template terminators belong to the outer user turn,
    not to the final semantic block, so they are excluded from the last block span.
    """
    _, end_token, _ = _get_block_scoring_config()

    matches = list(EXAMPLE_HEADER_RE.finditer(prompt))
    if len(matches) != expected_num_blocks:
        raise ValueError(
            f"Expected {expected_num_blocks} {globals().get('ITEM_WORD', 'Item')} blocks, but found {len(matches)}.\n"
            f"Prompt head:\n{prompt[:800]}"
        )

    spans = []
    for i, m in enumerate(matches):
        start = m.end()
        next_header_start = matches[i + 1].start() if i + 1 < len(matches) else len(prompt)

        if end_token:
            end_token_start = prompt.rfind(end_token, start, next_header_start)
            if end_token_start >= 0:
                end = end_token_start + len(end_token)
            else:
                end = next_header_start
        else:
            end = next_header_start
            if i + 1 == len(matches):
                end = _trim_trailing_chat_tail(prompt, start, end)
            while end > start and prompt[end - 1].isspace():
                end -= 1
        spans.append((start, end))

    return spans


def char_span_to_token_span(offset_mapping, char_start: int, char_end: int):
    """
    Convert character span [char_start, char_end) into token span [tok_start, tok_end).
    """
    token_ids = []
    for tok_idx, (s, e) in enumerate(offset_mapping):
        # Skip padding / special tokens / empty offsets.
        if s == 0 and e == 0:
            continue
        if e > char_start and s < char_end:
            token_ids.append(tok_idx)

    if not token_ids:
        raise ValueError(f"Could not map char span [{char_start}, {char_end}) to tokens.")

    return token_ids[0], token_ids[-1] + 1


def _get_inline_end_token_ids(end_token: str | None):
    """Token IDs for the explicit inline block terminator."""
    if not end_token:
        return []
    ids = tokenizer.encode(end_token, add_special_tokens=False)
    if not ids:
        raise ValueError(f"Could not tokenize BLOCK_END_TOKEN={end_token!r}")
    return [int(x) for x in ids]


def _ensure_span_includes_inline_end_token(prompt: str, token_span, end_token: str | None):
    """
    Some fast tokenizers report offset (0, 0) for special tokens such as
    <|im_end|> or <end_of_turn>. In that case char-offset mapping can end just
    before the explicit terminator. This helper checks token IDs and extends the
    span by the terminator sequence if needed.
    """
    if not end_token:
        return token_span

    tok_start, tok_end = token_span
    end_ids = _get_inline_end_token_ids(end_token)
    if not end_ids:
        return token_span

    enc_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    if isinstance(enc_ids[0], list):
        enc_ids = enc_ids[0]
    enc_ids = [int(x) for x in enc_ids]
    L = len(end_ids)

    if tok_end - tok_start >= L and enc_ids[tok_end - L:tok_end] == end_ids:
        return tok_start, tok_end

    if enc_ids[tok_end:tok_end + L] == end_ids:
        return tok_start, tok_end + L

    for shift in range(1, 5):
        if enc_ids[tok_end + shift:tok_end + shift + L] == end_ids:
            return tok_start, tok_end + shift + L

    return tok_start, tok_end


def _select_token_offsets_from_end(tok_start: int, tok_end: int, offsets_from_end):
    """
    Convert negative offsets from a full exemplar span into exact token indices.

    Offsets that run before tok_start are ignored. Thus [-1, -2, -3] behaves
    like the old last-3 selection, but automatically clips on very short spans.
    """
    if tok_end <= tok_start:
        raise ValueError(f"Empty token span [{tok_start}, {tok_end}) cannot be scored.")

    selected = []
    for offset in offsets_from_end:
        idx = int(tok_end) + int(offset)  # offset is negative: -1 -> tok_end - 1
        if tok_start <= idx < tok_end:
            selected.append(idx)

    if len(selected) == 0:
        raise ValueError(
            f"No selected offsets {offsets_from_end} fall inside token span [{tok_start}, {tok_end})."
        )

    return selected


def get_example_block_token_spans(prompt: str, offset_mapping, expected_num_blocks: int):
    """
    Return token selections for all Example blocks in the final rendered prompt.

    If a block terminator is configured, the terminator is included in the full
    exemplar span before token selection is applied. If not, the full span ends
    before the next Example header.

    Supported BLOCK_SCORE_MODE values:
      - "selected_offsets_from_end": select exact relative token positions from
        BLOCK_TOKEN_OFFSETS_FROM_END. For example, [-1] selects only the final
        token and [-2] selects only the second-to-final token.
      - legacy "last_k...": if BLOCK_TOKEN_OFFSETS_FROM_END is None, this is
        converted to [-1, -2, ..., -K] using BLOCK_LAST_K_TOKENS.
      - "All": return all tokens in each exemplar span.

    The function name is kept for backward compatibility; returned objects are
    TokenSelection instances that can still be unpacked as legacy (start, end)
    spans for logging/plotting.
    """
    mode, end_token, offsets_from_end = _get_block_scoring_config()
    char_spans = find_example_block_char_spans(
        prompt,
        expected_num_blocks=expected_num_blocks,
    )

    full_token_spans = [
        _ensure_span_includes_inline_end_token(
            prompt,
            char_span_to_token_span(offset_mapping, s, e),
            end_token=end_token,
        )
        for (s, e) in char_spans
    ]

    selections = []
    for block_idx, (tok_start, tok_end) in enumerate(full_token_spans, start=1):
        if mode == "all":
            indices = list(range(tok_start, tok_end))
        elif mode == "selected_offsets_from_end":
            indices = _select_token_offsets_from_end(tok_start, tok_end, offsets_from_end)
        else:
            # _get_block_scoring_config should already have validated the mode.
            raise ValueError(f"Unknown normalized BLOCK_SCORE_MODE={mode!r}")

        selections.append(
            TokenSelection(
                indices,
                full_start=tok_start,
                full_end=tok_end,
                label=f"{globals().get('ITEM_WORD', 'Item')} {block_idx}",
            )
        )

    return selections


def tokenize_prompt_for_offsets(prompt: str):
    """
    Prompts are already rendered with the intended BOS/chat special tokens, so
    add_special_tokens is disabled by default to avoid duplicating them.
    """
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=bool(globals().get("TOKENIZE_ADD_SPECIAL_TOKENS", False)),
        return_offsets_mapping=True,
        padding=False,
        truncation=False,
    )
    return enc


# Sanity check on one prompt.
_test_prompt = prompts_all_agnews[0][0]
_test_enc = tokenize_prompt_for_offsets(_test_prompt)
_test_offsets = _test_enc["offset_mapping"][0].tolist()
_test_spans = get_example_block_token_spans(
    prompt=_test_prompt,
    offset_mapping=_test_offsets,
    expected_num_blocks=NUM_BLOCKS_TOTAL,
)
print(f"{globals().get('ITEM_WORD', 'Item')} block token selections under current scoring mode:")
print("  PROMPT_FORMAT:", globals().get("PROMPT_FORMAT", "raw"))
print("  BLOCK_SCORE_MODE:", globals().get("BLOCK_SCORE_MODE", "selected_offsets_from_end"))
print("  BLOCK_END_TOKEN:", globals().get("BLOCK_END_TOKEN", None))
_printed_mode, _, _printed_offsets = _get_block_scoring_config()
if _printed_mode == "all":
    print("  BLOCK_TOKEN_OFFSETS_FROM_END: not used because BLOCK_SCORE_MODE='All'")
else:
    print("  BLOCK_TOKEN_OFFSETS_FROM_END:", _printed_offsets)
for i, sel in enumerate(_test_spans, start=1):
    s, e = token_selection_bounds(sel)
    idx = token_selection_to_indices(sel)
    toks = _test_enc["input_ids"][0, idx]
    print(
        f"  {globals().get('ITEM_WORD', 'Item')} {i}: envelope=[{s}, {e}) selected_n={len(idx)} "
        f"contiguous={sel.is_contiguous} decoded={tokenizer.decode(toks)!r}"
    )

In [ ]:
print(prompts_all_agnews[0][0])

In [ ]:

# ============================================================
# 5. First pass: select features by source-block activation frequency
# ============================================================

@torch.no_grad()
def first_pass_collect_previous_feature_frequency(
    model,
    tokenizer,
    sae,
    prompts_all_agnews,
    layer_idx: int,
    expected_num_blocks: int,
    source_block_index: int | None = None,
):
    """
    For every prompt:
      - capture layer output,
      - encode with SAE,
      - compute active-token frequency on the varied source block.

    The function name is kept for backward compatibility with earlier notebook
    versions, where the source block was always the immediately previous block.

    Returns:
      mean_prev_freq: np.ndarray [num_features]
      used_prompt_count: int
    """
    if source_block_index is None:
        source_block_index = int(LFH_SOURCE_BLOCK_INDEX)

    num_features = sae.n_features
    freq_sum = torch.zeros(num_features, dtype=torch.float64)
    used_prompt_count = 0

    for query_idx in range(len(prompts_all_agnews)):
        for variant_idx in range(len(prompts_all_agnews[query_idx])):
            prompt = prompts_all_agnews[query_idx][variant_idx]

            enc = tokenize_prompt_for_offsets(prompt)
            offset_mapping = enc["offset_mapping"][0].tolist()
            input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
            attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

            block_spans = get_example_block_token_spans(
                prompt=prompt,
                offset_mapping=offset_mapping,
                expected_num_blocks=expected_num_blocks,
            )

            source_span = block_spans[source_block_index]

            x = get_layer_output(
                model=model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                layer_idx=layer_idx,
            )  # [1, seq, hidden]

            sae_encoded = sae_activations(sae, x)[0].float().cpu()  # [seq, features]
            source_idx = token_selection_to_torch_indices(source_span, device=sae_encoded.device)
            source_freq = (sae_encoded.index_select(0, source_idx) > 0).float().mean(dim=0).double()

            freq_sum += source_freq
            used_prompt_count += 1

        print(f"first pass done query {query_idx + 1}/{len(prompts_all_agnews)}")

    mean_prev_freq = (freq_sum / max(used_prompt_count, 1)).numpy()
    return mean_prev_freq, used_prompt_count


mean_prev_freq, used_prompt_count = first_pass_collect_previous_feature_frequency(
    model=model,
    tokenizer=tokenizer,
    sae=sae,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=TARGET_LAYER,
    expected_num_blocks=NUM_BLOCKS_TOTAL,
    source_block_index=LFH_SOURCE_BLOCK_INDEX,
)

freq_over_threshold = {
    feat: float(freq)
    for feat, freq in enumerate(mean_prev_freq)
    if freq >= FREQ_THRESHOLD
}

print(f"Used {used_prompt_count} prompts.")
print(
    f"Selected {len(freq_over_threshold)} features with mean source-block "
    f"frequency >= {FREQ_THRESHOLD:.2f}"
)
print(f"Source block: Example {LFH_SOURCE_BLOCK_NUM}; target block: Example {LFH_TARGET_BLOCK_NUM}")


# ============================================================
# 6. Second pass: collect source/target mean SAE pre-activations
# ============================================================

@torch.no_grad()
def second_pass_collect_prev_tgt_logits(
    model,
    tokenizer,
    sae,
    prompts_all_agnews,
    layer_idx: int,
    expected_num_blocks: int,
    selected_feature_ids: list[int],
    source_block_index: int | None = None,
    target_block_index: int | None = None,
):
    """
    Collect for every prompt:
      - mean SAE pre-activation on the varied source block;
      - mean SAE pre-activation on the fixed target block.

    The returned variable names use prev/tgt for the configured source and target blocks.

    Returns:
      prev_logits_by_query: torch.Tensor [num_queries, num_variants, num_selected_features]
      tgt_logits_by_query : torch.Tensor [num_queries, num_variants, num_selected_features]
    """
    if source_block_index is None:
        source_block_index = int(LFH_SOURCE_BLOCK_INDEX)
    if target_block_index is None:
        target_block_index = int(LFH_TARGET_BLOCK_INDEX)

    num_queries = len(prompts_all_agnews)
    num_variants = len(prompts_all_agnews[0])
    selected_feature_ids_tensor = torch.tensor(selected_feature_ids, dtype=torch.long, device=sae.device)
    num_selected = len(selected_feature_ids)

    prev_logits_by_query = torch.zeros(num_queries, num_variants, num_selected, dtype=torch.float32)
    tgt_logits_by_query = torch.zeros(num_queries, num_variants, num_selected, dtype=torch.float32)

    for query_idx in range(num_queries):
        for variant_idx in range(num_variants):
            prompt = prompts_all_agnews[query_idx][variant_idx]

            enc = tokenize_prompt_for_offsets(prompt)
            offset_mapping = enc["offset_mapping"][0].tolist()
            input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
            attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

            block_spans = get_example_block_token_spans(
                prompt=prompt,
                offset_mapping=offset_mapping,
                expected_num_blocks=expected_num_blocks,
            )

            source_span = block_spans[source_block_index]
            tgt_span = block_spans[target_block_index]

            x = get_layer_output(
                model=model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                layer_idx=layer_idx,
            )  # [1, seq, hidden]

            logits = sae_logits_before_jumprelu(sae, x)[0]  # [seq, features]

            source_idx = token_selection_to_torch_indices(source_span, device=logits.device)
            tgt_idx = token_selection_to_torch_indices(tgt_span, device=logits.device)
            source_mean = logits.index_select(0, source_idx)[:, selected_feature_ids_tensor].mean(dim=0).detach().float().cpu()
            tgt_mean = logits.index_select(0, tgt_idx)[:, selected_feature_ids_tensor].mean(dim=0).detach().float().cpu()

            prev_logits_by_query[query_idx, variant_idx] = source_mean
            tgt_logits_by_query[query_idx, variant_idx] = tgt_mean

        print(f"second pass done query {query_idx + 1}/{num_queries}")

    return prev_logits_by_query, tgt_logits_by_query


selected_feature_ids = sorted(freq_over_threshold.keys())

prev_logits_by_query, tgt_logits_by_query = second_pass_collect_prev_tgt_logits(
    model=model,
    tokenizer=tokenizer,
    sae=sae,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=TARGET_LAYER,
    expected_num_blocks=NUM_BLOCKS_TOTAL,
    selected_feature_ids=selected_feature_ids,
    source_block_index=LFH_SOURCE_BLOCK_INDEX,
    target_block_index=LFH_TARGET_BLOCK_INDEX,
)

print("prev/source_logits_by_query:", tuple(prev_logits_by_query.shape))
print("tgt_logits_by_query        :", tuple(tgt_logits_by_query.shape))

In [ ]:

# ============================================================
# 7. Compute per-query Pearson correlations, then average
# ============================================================

def pearson_corr_per_feature_across_variants(prev_mat: np.ndarray, tgt_mat: np.ndarray):
    """
    prev_mat, tgt_mat: [num_variants, num_features]
    Returns:
      r: [num_features]
    """
    if prev_mat.shape != tgt_mat.shape:
        raise ValueError(f"Shape mismatch: prev={prev_mat.shape}, tgt={tgt_mat.shape}")

    x = prev_mat.astype(np.float64)
    y = tgt_mat.astype(np.float64)

    x = x - x.mean(axis=0, keepdims=True)
    y = y - y.mean(axis=0, keepdims=True)

    num = (x * y).sum(axis=0)
    den = np.sqrt((x * x).sum(axis=0) * (y * y).sum(axis=0))

    r = np.full(x.shape[1], np.nan, dtype=np.float64)
    valid = den > 0
    r[valid] = num[valid] / den[valid]
    return r


def compute_final_rho_results(
    prev_logits_by_query: torch.Tensor,
    tgt_logits_by_query: torch.Tensor,
    selected_feature_ids: list[int],
    sort_ascending: bool = True,
):
    """
    Compute:
      - per-query correlation for every selected feature,
      - average correlation across queries.

    Sort ascending, with the most negative correlations first.
    Negative correlation can be interpreted as a prediction-error-like
    or source-target local homeostasis effect.
    """
    prev_np = prev_logits_by_query.numpy()   # [Q, V, F]
    tgt_np = tgt_logits_by_query.numpy()     # [Q, V, F]

    num_queries = prev_np.shape[0]

    rho_by_query = np.stack(
        [
            pearson_corr_per_feature_across_variants(prev_np[q], tgt_np[q])
            for q in range(num_queries)
        ],
        axis=0,
    )  # [Q, F]

    querywise_rho = {}
    final_rho_results = {}

    for feat_local_idx, feat_id in enumerate(selected_feature_ids):
        vals = rho_by_query[:, feat_local_idx]
        querywise_rho[feat_id] = vals.tolist()

        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size == 0:
            continue
        final_rho_results[feat_id] = float(finite_vals.mean())

    final_rho_results = dict(
        sorted(final_rho_results.items(), key=lambda kv: kv[1], reverse=not sort_ascending)
    )

    return querywise_rho, final_rho_results, rho_by_query


querywise_rho, Final_rho_results, rho_by_query = compute_final_rho_results(
    prev_logits_by_query=prev_logits_by_query,
    tgt_logits_by_query=tgt_logits_by_query,
    selected_feature_ids=selected_feature_ids,
    sort_ascending=True,
)

print(f"Computed final correlations for {len(Final_rho_results)} features.")

rows = []
for rank, (feat, rho) in enumerate(Final_rho_results.items(), start=1):
    rows.append({
        "rank": rank,
        "feature_idx": int(feat),
        "source_block_num": int(LFH_SOURCE_BLOCK_NUM),
        "target_block_num": int(LFH_TARGET_BLOCK_NUM),
        "source_to_target_distance": int(LFH_TARGET_BLOCK_NUM - LFH_SOURCE_BLOCK_NUM),
        "mean_source_block_frequency": float(freq_over_threshold[feat]),
        # Backward-compatible column name for downstream analysis.
        "mean_prev_block_frequency": float(freq_over_threshold[feat]),
        "avg_querywise_pearson_r": float(rho),
    })

Final_rho_df = pd.DataFrame(rows)

print("=" * 100)
print("Top candidate LFH features, sorted by avg Pearson r ascending")
print(
    f"Source block Example {LFH_SOURCE_BLOCK_NUM} -> target block Example {LFH_TARGET_BLOCK_NUM}; "
    "most negative correlations first."
)
print("=" * 100)
display(Final_rho_df.head(50))

# Convenience defaults for plotting cells.
target_query_idx = 0
prev_demo_idx = 0
target_feature = int(Final_rho_df.iloc[0]["feature_idx"])
print("Default target_feature:", target_feature)

In [ ]:

# ============================================================
# 8. Scatter plot for one feature and one fixed target query
# ============================================================

import plotly.graph_objects as go
from scipy.stats import pearsonr

def plot_lfh_scatter(
    target_query_idx: int,
    target_feature: int,
    prev_logits_by_query,
    tgt_logits_by_query,
    selected_feature_ids,
):
    feature_local_idx = list(selected_feature_ids).index(target_feature)

    X = np.array(prev_logits_by_query[target_query_idx][:, feature_local_idx])
    Y = np.array(tgt_logits_by_query[target_query_idx][:, feature_local_idx])

    finite = np.isfinite(X) & np.isfinite(Y)
    X = X[finite]
    Y = Y[finite]

    if len(X) >= 2 and np.std(X) > 0 and np.std(Y) > 0:
        r, p = pearsonr(X, Y)
        slope, intercept = np.polyfit(X, Y, 1)
        x_line = np.linspace(X.min(), X.max(), 200)
        y_line = slope * x_line + intercept
    else:
        r, p = np.nan, np.nan
        x_line, y_line = np.array([]), np.array([])

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=X,
        y=Y,
        mode="markers",
        marker=dict(
            size=8,
            opacity=0.8,
            line=dict(width=1, color="black"),
        ),
        name="Prompt variants",
    ))

    if len(x_line) > 0:
        fig.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            line=dict(width=3),
            name="Linear fit",
        ))

    fig.update_layout(
        template="seaborn",
        title=dict(
            text=(
                f"Local Feature Homeostasis "
                f"({MODEL_SHORT_NAME}, Layer {TARGET_LAYER}, Feature {target_feature})<br>"
                f"corr={r:.3f}, p={p:.2e}"
            ),
            x=0.5,
            font=dict(family="Times New Roman", size=26),
        ),
        xaxis=dict(
            title=f"Activation on source demonstration (Example {LFH_SOURCE_BLOCK_NUM})",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=14),
        ),
        yaxis=dict(
            title=f"Activation on target demonstration (Example {LFH_TARGET_BLOCK_NUM})",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=14),
        ),
        height=600,
        width=900,
        showlegend=False,
    )

    fig.show()
    return fig, {"r": r, "p": p, "X": X, "Y": Y}


scatter_fig, scatter_stats = plot_lfh_scatter(
    target_query_idx=target_query_idx,
    target_feature=target_feature,
    prev_logits_by_query=prev_logits_by_query,
    tgt_logits_by_query=tgt_logits_by_query,
    selected_feature_ids=selected_feature_ids,
)

# Histograms

In [ ]:
# ============================================================
# 9. Histogram of average query-wise Pearson correlations
# ============================================================

import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde

def plot_rho_histogram(Final_rho_results, num_bins=25):
    values = np.array(list(Final_rho_results.values()), dtype=np.float64)
    values = values[np.isfinite(values)]
    N = len(values)

    if N == 0:
        raise ValueError("No finite rho values to plot.")

    min_val, max_val = float(np.min(values)), float(np.max(values))
    if min_val == max_val:
        min_val -= 1e-3
        max_val += 1e-3

    bin_width = (max_val - min_val) / num_bins

    fig = go.Figure()

    # Histogram with green bars
    fig.add_trace(go.Histogram(
        x=values,
        xbins=dict(start=min_val, end=max_val, size=bin_width),
        opacity=0.7,
        marker_color="green",
        name="Feature count",
    ))

    # KDE curve with dark green line
    if N >= 3 and np.std(values) > 0:
        kde = gaussian_kde(values)
        x_grid = np.linspace(-1, 1, 400)
        y_kde = kde(x_grid) * N * bin_width

        fig.add_trace(go.Scatter(
            x=x_grid,
            y=y_kde,
            mode="lines",
            line=dict(width=4, color="darkgreen"),
            name="KDE Curve",
        ))

    fig.update_layout(
        template="seaborn",
        barmode="overlay",
        title=dict(
            text=f"Local Feature Homeostasis Mechanisms in LLMs (Task: BFCL)<br>({MODEL_SHORT_NAME}, Layer {TARGET_LAYER})",
            x=0.5,
            font=dict(family="Times New Roman", size=26),
        ),
        xaxis=dict(
            title="Pearson correlation",
            range=[-1, 1],
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        yaxis=dict(
            title="Feature count",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        height=600,
        width=900,
        showlegend=False
    )

    # 4x resolution for the interactive 'Download Plot' button.
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'rho_histogram',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # Optional direct save at 4x resolution.
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("rho_histogram_highres.png", scale=4)

    return fig

# Uncomment to run if Final_rho_results, MODEL_SHORT_NAME, and TARGET_LAYER are defined
rho_hist_fig = plot_rho_histogram(Final_rho_results)